In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from scrape_functions import parse_bracket_data, scrape_espn_data, match_team_names
from constants import (
    DATA_FILENAME, TORVIK_FILE, KENPOM_FILE, ESPN_IDS,
    ORDER_OF_REGIONS, NAME_OVERRIDES,
)

pd.set_option('expand_frame_repr', False)
pd.set_option('display.max_columns', 0)
pd.set_option('display.max_rows', 68)

In [3]:
YEAR = 2026

In [4]:
# This needs to run after the first four games to get the correct names
GENDERS = ["mens", "womens"]
for gender in GENDERS:
    simple = parse_bracket_data(
        scrape_espn_data(id_=ESPN_IDS[YEAR][gender])
    )[['team_name', 'region_id', 'seed']]
    simple['team_region'] = simple['region_id'].apply(
        lambda x: dict(enumerate(ORDER_OF_REGIONS[gender]))[x - 1]
    )
    simple.to_csv(DATA_FILENAME.format(gender=gender), index=False)

# Add KenPom to Mens

In [5]:
kenpom = pd.read_csv(KENPOM_FILE.format(gender="mens"))
team_info = pd.read_csv(DATA_FILENAME.format(gender="mens"))

In [6]:
mapping, unmatched_ratings, unmatched_espn = match_team_names(
    kenpom, team_info['team_name'].tolist(), overrides=NAME_OVERRIDES["mens"]
)
print("Unmatched ratings:", unmatched_ratings)
print("Unmatched ESPN:   ", unmatched_espn)

Unmatched ratings: ['N.C. State', 'SMU', 'UMBC', 'Lehigh']
Unmatched ESPN:    []


In [7]:
# If any teams are unmatched above, add them to NAME_OVERRIDES["mens"] in constants.py and re-run.
kenpom['team_name'] = kenpom['Team'].map(mapping)
simple = kenpom.merge(team_info, left_on=['team_name', 'Seed'], right_on=['team_name', 'seed'])
simple = simple[['team_region', 'Seed', 'team_name', 'AdjustEM', 'AdjustT']]
assert len(simple) == 64, f"Expected 64 teams, got {len(simple)}"
simple.to_csv(DATA_FILENAME.format(gender="mens"), index=False)
simple

,team_region,Seed,team_name,AdjustEM,AdjustT
0,East,1,Duke,38.90,65.3
1,West,1,Arizona,37.66,69.8
2,Midwest,1,Michigan,37.59,70.9
3,South,1,Florida,33.79,70.5
4,South,2,Houston,33.43,63.3
5,Midwest,2,Iowa State,32.42,66.5
6,South,3,Illinois,32.10,65.5
7,West,2,Purdue,31.20,64.4
8,East,3,Michigan St,28.31,66.0
9,West,3,Gonzaga,28.10,68.6


# Add Torvik to Womens

In [8]:
kenpom = pd.read_csv(TORVIK_FILE.format(gender="womens"))
team_info = pd.read_csv(DATA_FILENAME.format(gender="womens"))

In [9]:
mapping, unmatched_ratings, unmatched_espn = match_team_names(
    kenpom, team_info['team_name'].tolist(), overrides=NAME_OVERRIDES["womens"]
)
print("Unmatched ratings:", unmatched_ratings)
print("Unmatched ESPN:   ", unmatched_espn)

Unmatched ratings: ['Richmond', 'Arizona St.', 'Stephen F. Austin', 'Samford']
Unmatched ESPN:    []


In [10]:
# If any teams are unmatched above, add them to NAME_OVERRIDES["womens"] in constants.py and re-run.
kenpom['team_name'] = kenpom['Team'].map(mapping)
simple = kenpom.merge(team_info, left_on=['team_name', 'Seed'], right_on=['team_name', 'seed'])
simple = simple[['team_region', 'Seed', 'team_name', 'pyth']]
assert len(simple) == 64, f"Expected 64 teams, got {len(simple)}"
simple.to_csv(DATA_FILENAME.format(gender="womens"), index=False)
simple

,team_region,Seed,team_name,pyth
0,Regional 1,1,UConn,0.999610
1,Regional 2,1,UCLA,0.999120
2,Regional 3,1,Texas,0.998830
3,Regional 4,1,South Carolina,0.998640
4,Regional 2,2,LSU,0.997750
5,Regional 3,2,Michigan,0.995260
6,Regional 2,3,Duke,0.993670
7,Regional 2,4,Minnesota,0.991480
8,Regional 4,2,Iowa,0.990790
9,Regional 1,2,Vanderbilt,0.990610
